In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
from bs_accel.data import generate_dataset, split_dataset
from bs_accel.model import build_mlp
from bs_accel.evaluate import load_trained_model
from bs_accel.train import save_checkpoint


# Call pricing from Black-Scholes

Samples use the package's fixed seed. Ten percent is held out for the final test;
validation uses ten percent of the remaining rows. All searches use validation loss.
Old outputs and scores were cleared because this split changes the experiment.


In [ ]:
X, y = generate_dataset("pricing")


In [ ]:
from math import exp

import optuna
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

X_train, X_test, y_train, y_test = split_dataset(X, y, "pricing")
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)


# Config

In [ ]:
import tensorflow as tf
print(tf.__version__, tf.test.is_built_with_cuda())

In [ ]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

# Architecture search on validation data


In [ ]:
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 2, 6)
    n_units = trial.suggest_int("n_units", 200, 600)
    batch_size = trial.suggest_int("batch_size", 1024, 2048)
    trial_model = build_mlp(X_train.shape[1], n_units, n_layers)
    history = trial_model.fit(
        X_train, y_train, epochs=1200, batch_size=batch_size, verbose=1,
        validation_data=(X_val, y_val),
        callbacks=[LearningRateScheduler(
            lambda epoch, lr: lr if epoch < 1000 else lr * exp(-0.1)
        )],
    )
    return history.history["val_loss"][-1]

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_params = study.best_params
print("Best architecture:", best_params, "validation MSE:", study.best_value)


In [ ]:
baseline_model = build_mlp(X_train.shape[1], 400, 5)
baseline_model.fit(
    X_train, y_train, epochs=1200, batch_size=1024, verbose=1,
    validation_data=(X_val, y_val),
    callbacks=[LearningRateScheduler(
        lambda epoch, lr: lr if epoch < 1000 else lr * exp(-0.1)
    )],
)
print("Baseline validation MSE:", baseline_model.evaluate(X_val, y_val, verbose=0))


In [ ]:
from time import perf_counter

import tf_keras as legacy_keras
import tensorflow_model_optimization as tfmot

# The pruning library uses legacy Keras; keep that model separate from Keras 3.
pruning_model = legacy_keras.Sequential([legacy_keras.layers.Input(shape=(X_train.shape[1],))])
for _ in range(5):
    pruning_model.add(legacy_keras.layers.Dense(400, activation="relu"))
pruning_model.add(legacy_keras.layers.Dense(1))
pruning_model.compile(optimizer="adam", loss="mse", jit_compile=False)
pruning_model.fit(
    X_train, y_train, epochs=1200, batch_size=1024, verbose=1,
    validation_data=(X_val, y_val),
    callbacks=[legacy_keras.callbacks.LearningRateScheduler(
        lambda epoch, lr: lr if epoch < 1000 else lr * exp(-0.1)
    )],
)
pruning_model.predict(X_val, verbose=0)
start = perf_counter()
pruning_model.predict(X_val, verbose=0)
before_seconds = perf_counter() - start

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(
    pruning_model,
    pruning_schedule=tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.5, final_sparsity=0.8,
        begin_step=2000, end_step=4000, frequency=100,
    ),
)
pruned_model.compile(optimizer="adam", loss="mse", jit_compile=False)
pruned_model.fit(
    X_train, y_train, epochs=1200, batch_size=1024, verbose=1,
    validation_data=(X_val, y_val),
    callbacks=[
        tfmot.sparsity.keras.UpdatePruningStep(),
        legacy_keras.callbacks.LearningRateScheduler(
            lambda epoch, lr: lr if epoch < 1000 else lr * exp(-0.1)
        ),
    ],
)
final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
final_pruned_model.compile(optimizer="adam", loss="mse", jit_compile=False)
print("Pruned validation MSE:", final_pruned_model.evaluate(X_val, y_val, verbose=0))
final_pruned_model.predict(X_val, verbose=0)
start = perf_counter()
final_pruned_model.predict(X_val, verbose=0)
after_seconds = perf_counter() - start
print(f"Warm validation inference: original={before_seconds:.6f}s pruned={after_seconds:.6f}s")


# Train phase one and save a complete checkpoint

The original architecture used 564 units, 4 hidden layers and batch size 1035.
This run uses the architecture selected on validation data.


In [ ]:
batch_size = best_params["batch_size"]
model = build_mlp(X_train.shape[1], best_params["n_units"], best_params["n_layers"])
model.fit(
    X_train, y_train, epochs=1200, batch_size=batch_size, verbose=1,
    validation_data=(X_val, y_val),
)
phase1_checkpoint = ROOT / "model" / "notebook-pricing" / "phase1.keras"
save_checkpoint(model, phase1_checkpoint)
print("Phase-one validation MSE:", model.evaluate(X_val, y_val, verbose=0))


# Learning-rate search from the same phase-one checkpoint


In [ ]:
def fit_phase2(model, exp_rate):
    model.fit(
        X_train, y_train, epochs=2000, batch_size=batch_size, verbose=1,
        validation_data=(X_val, y_val),
        callbacks=[
            LearningRateScheduler(lambda _epoch, lr: lr * exp(exp_rate)),
            EarlyStopping(monitor="val_loss", patience=80, restore_best_weights=True),
        ],
    )


def objective(trial):
    trial_model = load_trained_model(phase1_checkpoint)
    fit_phase2(trial_model, trial.suggest_float("exp_rate", -0.1, -0.001))
    return trial_model.evaluate(X_val, y_val, verbose=0)

learning_study = optuna.create_study(direction="minimize")
learning_study.optimize(objective, n_trials=50)
print("Best learning rate:", learning_study.best_params)
print("Validation MSE:", learning_study.best_value)


# Train the selected model and evaluate the test split once


In [ ]:
model = load_trained_model(phase1_checkpoint)
fit_phase2(model, learning_study.best_params["exp_rate"])
print("Final test MSE:", model.evaluate(X_test, y_test, verbose=0))

save_checkpoint(model, ROOT / "model" / "notebook-pricing" / "model.keras")

from statistics import median
from time import perf_counter

import numpy as np

from bs_accel.data import black_scholes
from bs_accel.evaluate import predict

predict(model, X_test)
ann_times, exact_times = [], []
for _ in range(3):
    start = perf_counter()
    ann_prices = predict(model, X_test)
    ann_times.append(perf_counter() - start)
    start = perf_counter()
    exact_prices = black_scholes(*X_test.T)
    exact_times.append(perf_counter() - start)
np.testing.assert_allclose(exact_prices, y_test, rtol=1e-12, atol=1e-14)
print(f"Warm inference median: ANN={median(ann_times):.6f}s exact={median(exact_times):.6f}s")
print("Use exact CPU pricing unless a target-hardware benchmark shows an ANN benefit.")
